# 02 — Supervised Segmentation: U-Net vs DeepLabV3+ (Colab)

The flagship experiment. Trains the **first mask-supervised segmentation model** on SteelDefectX,
both architectures head-to-head (same ImageNet ResNet encoder — fair comparison), 3 seeds each,
and reports IoU/F1-max/AUROC against the paper's zero-shot ceiling (37.49 IoU).

**Runtime:** GPU required — Runtime → Change runtime type → GPU (T4 is fine).

**Framing (do not drift):** *first supervised baseline vs the paper's zero-shot numbers* — never 'we beat their model'.

## 1. Install deps + clone your repo

In [ ]:
!pip install -q segmentation-models-pytorch albumentations torchmetrics huggingface_hub
# clone your code (or use Colab's git integration). Replace with your repo if pushing there.
import os
if not os.path.isdir('SteelDefectX'):
    !git clone https://github.com/d-mondal/SteelDefectX.git
%cd SteelDefectX

## 2. Download the dataset from Hugging Face (cached; first run ~a few min)

In [ ]:
from huggingface_hub import snapshot_download
DATA_ROOT = snapshot_download(repo_id='Zhaosxian/SteelDefectX', repo_type='dataset')
print('data at:', DATA_ROOT)

## 3. Make the FROZEN stratified split (once)

In [ ]:
import os
os.makedirs('data/splits', exist_ok=True)
!python -m src.data.make_split \
    --train-text {DATA_ROOT}/train-text.json \
    --out-dir data/splits --val-frac 0.15 --seed 42

## 4. Configure

In [ ]:
from src.config import Config
cfg = Config(
    data_root=DATA_ROOT,
    split_dir='data/splits',
    encoder='resnet34',
    batch_size=16,
    epochs=40,
    seeds=(0, 1, 2),
    use_wandb=False,      # set True + wandb.login() if you want tracking
)
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 5. (Optional) quick smoke test — 1 arch, 1 seed, 2 epochs

Run this FIRST to confirm the whole pipeline works end-to-end before committing to the full sweep.
It should complete in a few minutes and print a test-IoU already well above 37.49 if things are wired right.

In [ ]:
import dataclasses
from src.segmentation.train import train_one_run
smoke = dataclasses.replace(cfg, epochs=2, early_stop_patience=99)
m = train_one_run(smoke, arch='unet', seed=0)
print('smoke test IoU:', m['IoU'], '| Dice:', m['Dice'], '| AUROC:', m['AUROC'])

## 6. Full head-to-head: U-Net vs DeepLabV3+, 3 seeds each

6 training runs total. On a T4 this is roughly a few hours — keep the tab alive. Prints the
comparison table and saves `results/segmentation/comparison_summary.json`.

In [ ]:
from src.segmentation.train import run_comparison
results = run_comparison(cfg)

## 7. Per-class IoU (the new result the paper never reports)

Pulls the per-class breakdown from the best runs — watch the rarest classes (Rolled pit, Oxide scale of plate system, Crease).

In [ ]:
import json, glob
for f in sorted(glob.glob('results/segmentation/*_test_metrics.json')):
    m = json.load(open(f))
    print(f"\n{f.split('/')[-1]}  overall IoU={m['IoU']}")
    pc = sorted(m['per_class'].items(), key=lambda x: x[1]['IoU'])
    for c, v in pc[:5]:
        print(f"   worst: {c:32s} IoU={v['IoU']*100:5.1f}  (n={v['n']})")